# Agente RAG - Santos Pegasus Soluciones
Challenge Alura Agente (Oracle Next Education)

Este notebook construye un agente de IA que responde preguntas sobre los 5 documentos internos de **Santos Pegasus Soluciones**, usando:
- **LangChain** para orquestar el flujo RAG (Retrieval Augmented Generation)
- **PyPDF** para leer los PDFs
- **Chroma** como vector store (base de datos de embeddings)
- **Cohere** como modelo de embeddings y LLM

**Antes de empezar:** consigue tu API key gratuita en https://dashboard.cohere.com/api-keys


## 1. Instalar dependencias

In [ ]:
!pip install -q langchain langchain-community langchain-cohere cohere pypdf chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2

In [ ]:
!pip install -q -U langchain-text-splitters

## 2. Configurar la API key de Cohere

Se pide de forma segura (no queda escrita en el notebook).

In [ ]:
import os
from getpass import getpass

os.environ["COHERE_API_KEY"] = getpass("Pega tu Cohere API key: ")


Pega tu Cohere API key: ··········


## 3. Subir los PDFs

Sube aquí los 5 documentos de Santos Pegasus Soluciones:
- Manual de Onboarding para Nuevos Desarrolladores
- Guía Oficial de Ingeniería Back-end
- Guía Oficial de Ingeniería Front-end
- Protocolo de Respuesta a Incidentes y Post-Mortems
- Arquitectura de Microservicios y Mapa de Dominios


In [ ]:
from google.colab import files

uploaded = files.upload()  # selecciona los 5 PDFs
pdf_paths = list(uploaded.keys())
print("Archivos cargados:", pdf_paths)


Saving Arquitectura de Microservicios y Mapa de Dominios — Santo Pegasus Soluciones.pdf to Arquitectura de Microservicios y Mapa de Dominios — Santo Pegasus Soluciones.pdf
Saving Protocolo de Respuesta a Incidentes y Post-Mortems — Santo Pegasus Soluciones.pdf to Protocolo de Respuesta a Incidentes y Post-Mortems — Santo Pegasus Soluciones.pdf
Saving Guía Oficial de Ingeniería Front-end — Santo Pegasus Soluciones.pdf to Guía Oficial de Ingeniería Front-end — Santo Pegasus Soluciones.pdf
Saving Guía Oficial de Ingeniería Back-end.pdf to Guía Oficial de Ingeniería Back-end.pdf
Saving Manual de Onboarding para Nuevos Desarrolladores — Santo Pegasus Soluciones.pdf to Manual de Onboarding para Nuevos Desarrolladores — Santo Pegasus Soluciones.pdf
Archivos cargados: ['Arquitectura de Microservicios y Mapa de Dominios — Santo Pegasus Soluciones.pdf', 'Protocolo de Respuesta a Incidentes y Post-Mortems — Santo Pegasus Soluciones.pdf', 'Guía Oficial de Ingeniería Front-end — Santo Pegasus Soluc

## 4. Cargar y trocear (chunking) el contenido

Con ~150-200 páginas en total, no podemos mandar todo el texto al modelo de una vez. Lo dividimos en fragmentos pequeños y solapados.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    pages = loader.load()
    # Guardamos el nombre del archivo como metadata, útil para citar la fuente
    for p in pages:
        p.metadata["source_file"] = path
    all_docs.extend(pages)

print(f"Total de páginas cargadas: {len(all_docs)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunks = splitter.split_documents(all_docs)
print(f"Total de fragmentos (chunks) generados: {len(chunks)}")


Total de páginas cargadas: 152
Total de fragmentos (chunks) generados: 340


## 5. Generar embeddings y guardarlos en el vector store (Chroma)

In [ ]:
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = CohereEmbeddings(model="embed-multilingual-v3.0")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store listo.")


Vector store listo.


## 6. Construir la cadena RAG (retrieval + rerank + LLM Cohere)

In [ ]:
import cohere
from langchain_cohere import ChatCohere
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

co = cohere.Client(os.environ["COHERE_API_KEY"])
llm = ChatCohere(model="command-a-03-2025", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """Responde la pregunta basándote únicamente en el siguiente contexto extraído de los documentos internos de Santos Pegasus Soluciones.
Si la respuesta no está en el contexto, dilo claramente en vez de inventar.

Contexto:
{context}

Pregunta: {question}"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def retrieve_with_rerank(pregunta, fetch_k=40, top_n=8):
    """
    Paso 1: trae un grupo amplio de candidatos por similitud de embeddings (barato y rápido).
    Paso 2: usa Cohere Rerank para reordenarlos por relevancia REAL a la pregunta.
    Esto corrige el caso típico donde una pregunta formulada de forma natural
    (sin las palabras clave exactas del documento) no encuentra el chunk correcto
    solo con búsqueda vectorial.
    """
    candidatos = vectorstore.similarity_search(pregunta, k=fetch_k)
    if not candidatos:
        return []
    resultado = co.rerank(
        model="rerank-v3.5",
        query=pregunta,
        documents=[doc.page_content for doc in candidatos],
        top_n=min(top_n, len(candidatos)),
    )
    return [candidatos[r.index] for r in resultado.results]

def preguntar(pregunta):
    docs = retrieve_with_rerank(pregunta)
    contexto = format_docs(docs)
    respuesta = (prompt | llm | StrOutputParser()).invoke({"context": contexto, "question": pregunta})
    print("RESPUESTA:\n", respuesta)
    print("\nFUENTES:")
    for doc in docs:
        print("-", doc.metadata.get("source_file"), "| página:", doc.metadata.get("page"))


## 7. Probar el agente

Ajusta estas preguntas según el contenido real de tus PDFs.

In [ ]:
preguntar("¿Cuál es el protocolo a seguir ante un incidente de producción?")


RESPUESTA:
 Según el contexto proporcionado, el protocolo a seguir ante un incidente de producción en Santo Pegasus Soluciones se resume en los siguientes pasos:

1. **Detección**: Identificar el incidente, que es cualquier evento no planificado que cause o amenace con causar degradación o interrupción de los servicios en producción, con impacto real o potencial sobre los usuarios finales o los SLOs establecidos.

2. **Declaración**: El ingeniero on-call tiene hasta 15 minutos para declarar formalmente el incidente o descartarlo con justificación documentada en el canal `#incidents`. La declaración sigue un template específico que incluye detalles como el nivel de severidad (SEV-NIVEL), hora de detección, servicio afectado, impacto, usuarios afectados, métricas y el Incident Commander (IC) asignado.

3. **War Room**: Establecer una sala de guerra (War Room) para coordinar la respuesta al incidente.

4. **Diagnóstico**: Investigar la causa raíz del incidente para entender el problema su

In [ ]:
preguntar("¿Qué tecnologías se usan en el back-end según la guía oficial de ingeniería backend?")


RESPUESTA:
 Según la Guía Oficial de Ingeniería Back-end de Santo Pegasus Soluciones (versión 2.4.0), algunas de las tecnologías utilizadas en el back-end incluyen:

1. **Spring Boot**: Utilizado para exponer métricas de rendimiento y salud a través de Spring Boot Actuator.
2. **Micrometer**: Trabaja junto con Spring Boot Actuator para la exportación de métricas.
3. **Prometheus**: Recibe las métricas exportadas de manera nativa desde Spring Boot y Micrometer.
4. **Datadog**: Actúa como plataforma centralizada para la visualización, correlación y alertas basadas en las métricas.
5. **Spring Security**: Utilizado para la gestión centralizada de autenticación y autorización, con tokens JWT y flujo OAuth 2.0 / OpenID Connect.
6. **mTLS (Mutual TLS)**: Implementado para la comunicación segura entre servicios dentro de la VPC privada, con certificados gestionados por AWS Certificate Manager (ACM) Private CA.
7. **HashiCorp Vault, AWS Secrets Manager, Google Cloud Secret Manager**: Solucione

In [ ]:
preguntar("¿Qué pasos sigue un nuevo desarrollador en el proceso de onboarding?")


RESPUESTA:
 Según el contexto proporcionado, un nuevo desarrollador en Santos Pegasus Soluciones sigue los siguientes pasos en el proceso de onboarding:

1. **Completar el checklist de onboarding de la Semana 1** (SECCIÓN 12), que incluye:
   - Verificar que el correo corporativo (@santopegasus.com) esté funcionando.
   - Instalar Slack y unirse a los canales obligatorios.
   - Confirmar el acceso a GitHub (organización privada).

2. **Configurar el entorno local** (SECCIÓN 4 para Back-end y SECCIÓN 5 para Front-end), aunque no se detallan los pasos específicos en el contexto proporcionado.

3. **Leer la Guía de Ingeniería Back-end o Front-end completa** (SECCIÓN 7.1).

4. **Resolver 2-3 tickets de tipo `good-first-issue` en Jira con apoyo del buddy** (SECCIÓN 7.1).

5. **Participar en al menos 1 Code Review como observador** (SECCIÓN 7.1).

6. **Entregar features pequeñas de forma autónoma (con revisión del buddy) y escribir pruebas unitarias para el código propio** (SECCIÓN 7.1, 30-6

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('chroma_db', 'zip', './chroma_db')
files.download('chroma_db.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>